In [1]:
!pip install roboflow -q

from roboflow import Roboflow
rf = Roboflow(api_key="Qnj3uwQmrFh75CdE6blG") # Consider resetting this key in your dashboard later!
project = rf.workspace("7gamil").project("lp7510-numbers-roi-ocr-gia7i")
version = project.version(4)
dataset = version.download("yolov8")

print(f"✅ Dataset downloaded to: {dataset.location}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to LP7510-Numbers-ROI-+-OCR-4 in yolov8:: 100%|██████████| 65327/65327 [00:09<00:00, 7003.15it/s] 


✅ Dataset downloaded to: /kaggle/working/LP7510-Numbers-ROI-+-OCR-4


In [2]:
# YOLOv8
!pip install -q ultralytics

# PARSeq's runtime deps (hubconf.py pulls these in transitively)
!pip install -q timm==0.6.13 pytorch-lightning lmdb nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 7.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 16.2 MB/s eta 0:00:00


In [3]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from glob import glob
from torchvision import transforms as T
from ultralytics import YOLO

# ---- CONFIG ----
YOLO_WEIGHTS_PATH = "/kaggle/input/datasets/princemridul/pt-ckpt/best_yolo.pt"   

# dynamically pulling the path from the Roboflow download cell!
VALID_IMAGES_DIR  = os.path.join(dataset.location, "valid", "images")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_CONF_THRESHOLD = 0.25

print(f"✅ Using device: {DEVICE}")
print(f"✅ Looking for images in: {VALID_IMAGES_DIR}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Using device: cpu
✅ Looking for images in: /kaggle/working/LP7510-Numbers-ROI-+-OCR-4/valid/images


In [4]:
def load_yolo_model(weights_path: str) -> YOLO:
    return YOLO(weights_path)


def load_pretrained_parseq(device: str):
    """
    Loads the official pretrained PARSeq model via torch.hub.
    Downloads repo + weights automatically (requires internet on Kaggle).
    """
    model = torch.hub.load('baudm/parseq', 'parseq', pretrained=True, trust_repo=True)
    model = model.eval().to(device)

    # Standard PARSeq preprocessing pipeline (matches strhub's SceneTextDataModule).
    # Hand-rolled here so we don't depend on cloning the repo separately / sys.path hacks.
    img_size = model.hparams.img_size  # typically (32, 128)
    transform = T.Compose([
        T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(0.5, 0.5),
    ])
    return model, transform


print("Loading YOLOv8...")
yolo_model = load_yolo_model(YOLO_WEIGHTS_PATH)

print("Loading pretrained PARSeq (downloading on first run)...")
parseq_model, parseq_transform = load_pretrained_parseq(DEVICE)

print("Both models loaded successfully.")

Loading YOLOv8...
Loading pretrained PARSeq (downloading on first run)...
Downloading: "https://github.com/baudm/parseq/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://github.com/baudm/parseq/releases/download/v1.0.0/parseq-bb5792a6.pt" to /root/.cache/torch/hub/checkpoints/parseq-bb5792a6.pt


100%|██████████| 91.0M/91.0M [00:01<00:00, 76.9MB/s]


Both models loaded successfully.


In [5]:
def detect_roi(image_bgr: np.ndarray, yolo_model: YOLO, conf_threshold: float = 0.25):
    results = yolo_model.predict(source=image_bgr, conf=conf_threshold, verbose=False)

    if len(results) == 0 or results[0].boxes is None or len(results[0].boxes) == 0:
        return None

    boxes = results[0].boxes
    best_idx = boxes.conf.argmax().item()
    xyxy = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    conf = boxes.conf[best_idx].item()
    return xyxy, conf


def crop_image(image_bgr: np.ndarray, xyxy: np.ndarray) -> np.ndarray:
    x1, y1, x2, y2 = xyxy
    h, w = image_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    return image_bgr[y1:y2, x1:x2]


def recognize_text(crop_bgr: np.ndarray, parseq_model, img_transform, device: str):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(crop_rgb)

    tensor = img_transform(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = parseq_model(tensor)
        probs = logits.softmax(-1)
        labels, confidences = parseq_model.tokenizer.decode(probs)

    predicted_text = labels[0]
    mean_conf = confidences[0].mean().item() if len(confidences[0]) > 0 else 0.0
    return predicted_text, mean_conf


def run_pipeline(image_path: str, yolo_model, parseq_model, img_transform, device: str,
                  conf_threshold: float = 0.25) -> dict:
    result = {
        "image": os.path.basename(image_path),
        "status": "OK",
        "yolo_conf": None,
        "predicted_text": None,
        "parseq_conf": None,
        "error": None,
    }

    try:
        image_bgr = cv2.imread(image_path)
        if image_bgr is None:
            raise ValueError(f"cv2 could not read image at {image_path}")

        detection = detect_roi(image_bgr, yolo_model, conf_threshold)
        if detection is None:
            result["status"] = "NO_DETECTION"
            result["error"] = "YOLO found no bounding box above threshold"
            return result

        xyxy, yolo_conf = detection
        result["yolo_conf"] = round(yolo_conf, 4)

        crop = crop_image(image_bgr, xyxy)
        if crop.size == 0:
            result["status"] = "EMPTY_CROP"
            result["error"] = "Bounding box produced an empty crop"
            return result

        text, parseq_conf = recognize_text(crop, parseq_model, img_transform, device)
        result["predicted_text"] = text
        result["parseq_conf"] = round(parseq_conf, 4)

    except Exception as e:
        result["status"] = "ERROR"
        result["error"] = str(e)

    return result

In [6]:
test_image = sorted(glob(os.path.join(VALID_IMAGES_DIR, "*")))[0]
print(run_pipeline(test_image, yolo_model, parseq_model, parseq_transform, DEVICE))

{'image': 'outputFile_1000_png.rf.392dd06ea382d821235f6a374d2dca25.jpg', 'status': 'OK', 'yolo_conf': 0.8989, 'predicted_text': '782700', 'parseq_conf': 0.8549, 'error': None}


In [7]:
def benchmark(image_dir: str, n_images: int, yolo_model, parseq_model, img_transform, device: str):
    image_paths = sorted(glob(os.path.join(image_dir, "*.jpg")) +
                          glob(os.path.join(image_dir, "*.png")) +
                          glob(os.path.join(image_dir, "*.jpeg")))[:n_images]

    if not image_paths:
        print(f"No images found in {image_dir}")
        return pd.DataFrame()

    records = []
    for path in image_paths:
        res = run_pipeline(path, yolo_model, parseq_model, img_transform, device)
        records.append(res)

    return pd.DataFrame(records)


results_df = benchmark(VALID_IMAGES_DIR, 20, yolo_model, parseq_model, parseq_transform, DEVICE)

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)
print(results_df[["image", "status", "yolo_conf", "predicted_text", "parseq_conf"]].to_string(index=False))

n_total = len(results_df)
n_ok = (results_df["status"] == "OK").sum()
print(f"\nSuccess rate: {n_ok}/{n_total} ({n_ok/n_total*100:.1f}%)")

                                                      image status  yolo_conf predicted_text  parseq_conf
outputFile_1000_png.rf.392dd06ea382d821235f6a374d2dca25.jpg     OK     0.8989         782700       0.8549
outputFile_1001_png.rf.b49a0afee2eb84d236ab7ccbd0d2dcc1.jpg     OK     0.8976         82.800       0.9892
outputFile_1002_png.rf.a1004a376212541b167b4da25565c3e8.jpg     OK     0.8974         82.980       0.9917
outputFile_1004_png.rf.71a6908eba85910b3e5e48b10679d428.jpg     OK     0.8955        783.200       0.9488
outputFile_1005_png.rf.9626453c8cfd8fee57f1fde14a67fd45.jpg     OK     0.8978         83.350       0.9948
outputFile_1006_png.rf.6cfbb5b6ba84dc50cf00e9af3c1d2179.jpg     OK     0.8980         83.500       0.9857
outputFile_1007_png.rf.43acf286fae69724a81052137ad0ae1d.jpg     OK     0.8993         83.550       0.9930
outputFile_1008_png.rf.e394dd5f07c6a963517e29656a107221.jpg     OK     0.9011         83.650       0.9918
outputFile_1009_png.rf.68f0de2855f70ad8c4a0759

In [8]:
import pandas as pd
from tqdm import tqdm

test_images = sorted(glob(os.path.join(VALID_IMAGES_DIR, "*")))[:50]

results_list = []

print("🚀 Running Pipeline...")
for img_path in tqdm(test_images):
    res = run_pipeline(img_path, yolo_model, parseq_model, parseq_transform, DEVICE)
    results_list.append(res)
    
df_results = pd.DataFrame(results_list)

display(df_results.head(20))

🚀 Running Pipeline...


100%|██████████| 50/50 [00:14<00:00,  3.52it/s]


,image,status,yolo_conf,predicted_text,parseq_conf,error
0,outputFile_1000_png.rf.392dd06ea382d821235f6a3...,OK,0.8989,782700,0.8549,None
1,outputFile_1001_png.rf.b49a0afee2eb84d236ab7cc...,OK,0.8976,82.800,0.9892,None
2,outputFile_1002_png.rf.a1004a376212541b167b4da...,OK,0.8974,82.980,0.9917,None
3,outputFile_1004_png.rf.71a6908eba85910b3e5e48b...,OK,0.8955,783.200,0.9488,None
4,outputFile_1005_png.rf.9626453c8cfd8fee57f1fde...,OK,0.8978,83.350,0.9948,None
5,outputFile_1006_png.rf.6cfbb5b6ba84dc50cf00e9a...,OK,0.8980,83.500,0.9857,None
6,outputFile_1007_png.rf.43acf286fae69724a810521...,OK,0.8993,83.550,0.9930,None
7,outputFile_1008_png.rf.e394dd5f07c6a963517e296...,OK,0.9011,83.650,0.9918,None
8,outputFile_1009_png.rf.68f0de2855f70ad8c4a0759...,OK,0.8999,83.750,0.9907,None
9,outputFile_1010_png.rf.5910e9dfbd9caca646f882b...,OK,0.9026,83.850,0.9899,None


In [9]:
!pip install Levenshtein -q
import Levenshtein

def calculate_ocr_metrics(predictions, ground_truths):
    if len(predictions) != len(ground_truths):
        print(f"❌ Error: {len(predictions)} predictions vs {len(ground_truths)} ground truths. They must match!")
        return
    
    exact_matches = 0
    total_chars = 0
    total_errors = 0

    for pred, gt in zip(predictions, ground_truths):
        pred_str = str(pred).strip()
        gt_str = str(gt).strip()
        
        # Exact Match
        if pred_str == gt_str:
            exact_matches += 1
            
        # Character Error Rate (CER)
        distance = Levenshtein.distance(pred_str, gt_str)
        total_errors += distance
        total_chars += len(gt_str)

    em_accuracy = (exact_matches / len(predictions)) * 100
    cer = (total_errors / total_chars) * 100 if total_chars > 0 else 0


    print(f" Exact Match Accuracy: {em_accuracy:.2f}% ({exact_matches}/{len(predictions)})")
    print(f" Character Error Rate (CER): {cer:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.6 MB/s eta 0:00:00


In [10]:
ground_truths = [
    '82.700', '82.800', '82.980', '83.200', '83.350', 
    '83.500', '83.550', '83.650', '83.750', '83.850'
]

predictions = df_results['predicted_text'].tolist()[:10]

calculate_ocr_metrics(predictions, ground_truths)

 Exact Match Accuracy: 80.00% (8/10)
 Character Error Rate (CER): 5.00%
